# Walkthrough — FLTA 2026 evaluation companion artefact

**Federated learning only.** This evaluation tests the privacy card framework against a cross-device federated-learning deployment on the MedMNIST v2 **BloodMNIST** benchmark — a real, open, peer-reviewed medical-imaging dataset [Yang et al., *Sci. Data* 2023].

## The four study questions

- **SQ-1 — calibration soundness.** Does the card's `risk_calibration.attack_target` claim match the empirical attack rates measured by the harness? Two FL-native attacks: per-record MIA against the released model [Carlini et al., S&P 2022]; gradient inversion against per-round client updates [Geiping et al., NeurIPS 2020].
- **SQ-2 — per-subject metadata fidelity.** Does the rule engine include / exclude subjects correctly on the basis of pod-resident consent receipts, withdrawal logs, jurisdictional tags, and **FL participation logs**?
- **SQ-3 — composability surfacing.** Do the FLTA composability rules fire on mis-configured chains and stay silent on correctly-configured ones, *against a hand-authored expectations manifest*?
- **SQ-5 — scalar vs stepwise reporting.** Side-by-side, two FL configurations: what does scalar (ε, AUROC) reporting *say*, and what does the attack-rate-calibrated stepwise report *show*?

(SQ-4 — reviewer legibility — is a workshop study under `workshops/`, not a notebook.)

In [1]:
import json, sys
from pathlib import Path

EVAL_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(EVAL_DIR))
print(f'EVAL_DIR = {EVAL_DIR}')

EVAL_DIR = /Users/john.smith/Documents/dev/stepwise-privacy-cards


In [2]:
if not (EVAL_DIR / 'data' / '_manifest.json').exists():
    import subprocess
    subprocess.run([sys.executable, str(EVAL_DIR / 'scripts' / 'prepare.py')], check=True)

data_manifest = json.loads((EVAL_DIR / 'data' / '_manifest.json').read_text())
print(f"dataset       : {data_manifest['dataset']}")
print(f"citation      : {data_manifest['citation']}")
print(f"n_train       : {data_manifest['n_train']}")
print(f"n_test        : {data_manifest['n_test']}")
print(f"n_classes     : {data_manifest['n_classes']}")
print(f"input_dim     : {data_manifest['input_dim']} (28*28*3 = 2352)")
print(f"npz sha256    : {data_manifest['npz_sha256'][:16]}…")
print(f"partition α   : {data_manifest['partition_alpha']} (Dirichlet non-IID, Hsu et al. 2019)")
print(f"n pods (data) : {data_manifest['n_pods']}")
sizes = sorted(data_manifest['pod_sizes'])
print(f"pod sizes     : min={sizes[0]}, median={sizes[len(sizes)//2]}, max={sizes[-1]}")

dataset       : bloodmnist
citation      : Yang et al., Sci. Data 2023; Acevedo et al., Data in Brief 2020 (underlying dataset)
n_train       : 11959
n_test        : 3421
n_classes     : 8
input_dim     : 2352 (28*28*3 = 2352)
npz sha256    : 062023e186f537e2…
partition α   : 0.5 (Dirichlet non-IID, Hsu et al. 2019)
n pods (data) : 50
pod sizes     : min=48, median=206, max=937


In [3]:
card = json.loads((EVAL_DIR / 'card' / 'example_chain.json').read_text())
print(f"card_id        : {card['card_id']}")
print(f"PETs           : {[c['primitive_id'] for c in card['pet_components']]}")
print(f"jurisdictions  : {card['jurisdictional_context']}")
print()
print('stepwise_chain:')
for step in card['stepwise_chain']:
    print(f"  step {step['step']}: {step['name']:45s} adds {step['pet_added']:6s} EP={step['exposure_problem_ref']}")
print()
rc = card['risk_calibration']
at = rc['attack_target']
print('risk_calibration.attack_target:')
print(f"  attack          : {at['attack']}")
print(f"  target_advantage: {at['target_advantage']}")
print(f"  target_fpr      : {at['target_fpr']}")
print(f"  evidence_ref    : {at['evidence_ref']}")

card_id        : CARD-flta-fl-bloodmnist-202605
PETs           : ['FL', 'TEE', 'DP-C']
jurisdictions  : {'data_subjects': ['EU'], 'controllers': ['EU'], 'operators': ['EU'], 'cross_border_mechanism': None, 'notes': 'Coherent single-regime deployment: all participants are EU residents; controller and operator are EU. No cross-border mechanism required at this configuration. A future US-region operator step would require SCCs or adequacy and would trigger COMP-JURIS-001 until declared.'}

stepwise_chain:
  step 0: Baseline — completely private                 adds none   EP=baseline
  step 1: Authorise FL clients to read pod-resident data adds FL     EP=EP-04
  step 2: Bound per-update leakage via central DP       adds DP-C   EP=EP-04
  step 3: Protect aggregation environment               adds TEE    EP=EP-03

risk_calibration.attack_target:
  attack          : MIA
  target_advantage: 0.03
  target_fpr      : 0.001
  evidence_ref    : results/sq1/mia_per_record__B__bloodmnist.json


In [4]:
pods_manifest = json.loads((EVAL_DIR / 'pods' / '_manifest.json').read_text())
print(f"federation size       : {pods_manifest['size']}")
print(f"positive / negative   : {pods_manifest['positive_count']} / {pods_manifest['negative_count']}")
print('negative distribution:')
for mode, count in pods_manifest['negative_distribution'].items():
    print(f"    {mode:25s} {count}")
print()
print('Sample positive pod resources (persona_pos_000):')
for res in ['consent.jsonld', 'jurisdiction.jsonld', 'participation.jsonld']:
    obj = json.loads((EVAL_DIR / 'pods' / 'persona_pos_000' / res).read_text())
    print(f'  {res:25s} keys: {list(obj.keys())}')

federation size       : 200
positive / negative   : 100 / 100
negative distribution:
    bad_signature             15
    controller_mismatch       10
    expired_consent           25
    lifecycle_violation       15
    missing_jurisdiction      15
    withdrawn                 20

Sample positive pod resources (persona_pos_000):
  consent.jsonld            keys: ['@context', '@type', 'controller', 'issued', 'lawful_basis', 'not_after', 'purposes', 'signature', 'subject_webid', 'withdrawable']
  jurisdiction.jsonld       keys: ['@type', 'controller_jurisdiction', 'cross_border_mechanism', 'data_subject_jurisdiction', 'operator_jurisdiction']
  participation.jsonld      keys: ['@context', '@type', 'entries', 'last_round', 'subject_webid']


In [5]:
battery = json.loads((EVAL_DIR / 'card' / 'battery_manifest.json').read_text())
expected = json.loads((EVAL_DIR / 'card' / 'battery_expected.json').read_text())
exp_map = {e['chain_id']: e['expected_firings'] for e in expected['chains']}
for entry in battery['entries']:
    firings = exp_map.get(entry['chain_id'], [])
    tag = ', '.join(f.removeprefix('COMP-') for f in firings) if firings else 'none (well-configured)'
    print(f"  {entry['chain_id']:42s} expected: {tag}")

  FL__H1_default                             expected: SECAGG-001
  FL__H2_gradinv_eps6                        expected: SECAGG-001
  FL__H3_juris_unmechanised                  expected: JURIS-001, SECAGG-001
  FL__H4_well_configured                     expected: none (well-configured)
  FL+DP__H1_default                          expected: GRADINV-001, SECAGG-001
  FL+DP__H2_gradinv_eps6                     expected: GRADINV-001, SECAGG-001
  FL+DP__H3_juris_unmechanised               expected: GRADINV-001, JURIS-001, SECAGG-001
  FL+DP__H4_well_configured                  expected: none (well-configured)
  FL+TEE+DP__H1_default                      expected: GRADINV-001, SECAGG-001, TEEMPC-001
  FL+TEE+DP__H2_gradinv_eps6                 expected: GRADINV-001, SECAGG-001, TEEMPC-001
  FL+TEE+DP__H3_juris_unmechanised           expected: GRADINV-001, JURIS-001, SECAGG-001, TEEMPC-001
  FL+TEE+DP__H4_well_configured              expected: none (well-configured)
  FL+MPC+DP__H1_default  

## Next steps

Run the study-question notebooks in any order. Each is self-contained and writes a result record under `results/<sq>/`.

- [`sq1-calibration/01_mia_per_record_sweep.ipynb`](sq1-calibration/01_mia_per_record_sweep.ipynb) — per-record MIA against the FL-released model
- [`sq1-calibration/02_gradient_inversion.ipynb`](sq1-calibration/02_gradient_inversion.ipynb) — gradient inversion against per-round client updates
- [`sq2-metadata/01_per_subject_fidelity.ipynb`](sq2-metadata/01_per_subject_fidelity.ipynb) — rule engine vs the 200-pod federation
- [`sq3-composability/01_rule_battery.ipynb`](sq3-composability/01_rule_battery.ipynb) — rule engine vs hand-authored battery expectations
- [`sq5-comparison/01_scalar_vs_stepwise.ipynb`](sq5-comparison/01_scalar_vs_stepwise.ipynb) — head-to-head reporting comparison